In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/titanic/train.csv
/kaggle/input/competitions/titanic/test.csv
/kaggle/input/competitions/titanic/gender_submission.csv


In [2]:
import os #brings python's built-in toolkit for interacting with the system
for dirname, _, filenames in os.walk('/kaggle/input'): #looks into /kaggle/input, walks through every folder inside it, for each folder it gives you the folder's path, subfolders inside it (don't need, hence underscore), list of files inside it
    for filename in filenames: #goes through each file in that folder
        print(os.path.join(dirname, filename)) #displays folder path and filename into one full path

/kaggle/input/competitions/titanic/train.csv
/kaggle/input/competitions/titanic/test.csv
/kaggle/input/competitions/titanic/gender_submission.csv


In [3]:
import pandas as pd #imports pandas and is referred to as pd
import numpy as np ##imports numpy and is referred to as np

train = pd.read_csv('/kaggle/input/competitions/titanic/train.csv')
test = pd.read_csv('/kaggle/input/competitions/titanic/test.csv') 
#reads a CSV file from exact path, loads it into a DataFram

train.head() #displays firts 5 rows of train
test.head() #same for test

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


In [4]:
#calculates percent of women who survived
women = train.loc[train.Sex == 'female']["Survived"]
rate_women = sum(women)/len(women)

print("% of women who survived:", rate_women)

% of women who survived: 0.7420382165605095


In [5]:
#same but for men
men = train.loc[train.Sex == 'male']["Survived"]
rate_men = sum(men)/len(men)

print("% of men who survived:", rate_men)

% of men who survived: 0.18890814558058924


In [6]:
train['Age'] = train['Age'].fillna(train['Age'].median()) #'fillna' is a pandas function that fills in missing values, this line fills missing Age with the median age
test['Age'] = test['Age'].fillna(test['Age'].median()) #does the same but for test data

train['Embarked'] = train['Embarked'].fillna(train['Embarked'].mode()[0]) #Embarked shows which port each passenger boarded from, mode returns a small pandas Series (in case there's a link between two equally common values) [0] grabs the first item from the list
test['Fare'] = test['Fare'].fillna(test['Fare'].median()) #does the same as the second line in this section but for fare - the ticket price

#the cabin has too many missing values - therefore, we drop it 
train = train.drop('Cabin', axis=1)
test = test.drop('Cabin', axis=1)

In [7]:
#extracting the title of a person from their name
for dataset in [train, test]: #does the following steps for both the train and test table
    dataset['Title'] = dataset['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False) #uses regular expression to pull out the title - word right before full stop after a space
    dataset['Title'] = dataset['Title'].replace(
        ['Lady', 'Countess', 'Capt', 'Col', 'Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare'
    ) #takes unusual titles and puts them together into a rare category
    dataset['Title'] = dataset['Title'].replace(['Mlle', 'Ms'], 'Miss') #replaces french versions with Miss
    dataset['Title'] = dataset['Title'].replace(['Mme'], 'Mrs') #same but for Mrs

for dataset in [train, test]:
    dataset['FamilySize'] = dataset['SibSp'] + dataset['Parch'] + 1 #adds number of siblings/spouse to number of parents/children on board, adding one gives total family size

In [8]:
print(train.columns.tolist())

['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Embarked', 'Title', 'FamilySize']


In [9]:
#encoding the categorical variables so models can use them 
train = pd.get_dummies(train, columns=['Sex', 'Embarked', 'Title'], drop_first= True) #turns each category into yes/no (1/0)
test = pd.get_dummies(test, columns=['Sex', 'Embarked', 'Title'], drop_first= True) #same but for test data

#the test set might end up missing the title column that the train set has, so align them
train, test = train.align (test, join='left', axis=1, fill_value=0) #safety step, get_dummies creates collumns based on what categories actually appear in each data set; align makes sure both tables end up with the same set of collumns filling in 0 for anything that is missing
test = test.drop('Survived', axis=1, errors = 'ignore') #cleanup step, survived only exists in training data, so remove it

train.head()

,PassengerId,Survived,Pclass,Name,Age,SibSp,Parch,Ticket,Fare,FamilySize,Sex_male,Embarked_Q,Embarked_S,Title_Miss,Title_Mr,Title_Mrs,Title_Rare
0,1,0,3,"Braund, Mr. Owen Harris",22.0,1,0,A/5 21171,7.2500,2,True,False,True,False,True,False,False
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",38.0,1,0,PC 17599,71.2833,2,False,False,False,False,False,True,False
2,3,1,3,"Heikkinen, Miss. Laina",26.0,0,0,STON/O2. 3101282,7.9250,1,False,False,True,True,False,False,False
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",35.0,1,0,113803,53.1000,2,False,False,True,False,False,True,False
4,5,0,3,"Allen, Mr. William Henry",35.0,0,0,373450,8.0500,1,True,False,True,False,True,False,False


In [10]:
#split data for training and validation
from sklearn.model_selection import train_test_split #imports train_test_split from sklearn.model_selection - standard python library for machine learning

pred_columns = ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'FamilySize', 
                'Sex_male', 'Embarked_Q', 'Embarked_S', 'Title_Miss', 
                'Title_Mr', 'Title_Mrs', 'Title_Rare'] #list of column names 

X = train[pred_columns] #inputs to predict y, X will be a table with 13 columns
y = train['Survived'] #survived column, 1=survived, 0=died

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42) #divides data 80% to X_train and y_train, 20% to X_test and y_test

In [11]:
#training and comparing 3 models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score  #4 imports, imports 3 model types to be compared and the accuracy score tool to measure how good predictions are 
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000), #gives logistic regression more attempts as its default limit can be low 
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42) #random_state=42 gives these 2 models internal randomness reproducible
} #creates a dictionary giving the actual model object a readable name 
for name, model in models.items(): #loops through dictionary one item at a time 
    model.fit(X_train, y_train) #training step, learns pattern connecting X and y training data
    predictions = model.predict(X_test) #trained model is given validation inputs, asked to guess survival for each one, stored as predictions
    accuracy = accuracy_score(y_test, predictions) #compares predictions against real values, calculates percentage correct
    print(f"{name}: {accuracy:.4f}") #prints the model name and accuracy score, ':.4f' controls formatting

Logistic Regression: 0.7933
Decision Tree: 0.7654
Random Forest: 0.8324


In [12]:
from sklearn.model_selection import GridSearchCV #imports new tool, automates process of trying out different settings for a model and finding which contribution is the best fit

parameter_grid = { 
    'n_estimators': [100, 200, 300], #how many decision trees to build inside the forest, more trees = better accuracy, slower training
    'max_depth': [None, 5, 10], #how many levels each tree us allowed to grow, none = no limit
    'min_samples_split': [2,5,10] #minimum number of data points a group must have before a tree is allowed to split further 
} #creates a dictionary for how the random forest builds

grid_search = GridSearchCV(RandomForestClassifier(random_state=42), parameter_grid, cv=5, scoring='accuracy')
#searches the base model type, param_grid - the settings to try, cv=5 - 5-cross validation (splits training data into 5 equal sections, trains on 4 of them and tests on the fifth then repeats 5 times and averages the results, gives more reliable estimate,), scoring='accuracy' - judge each combination by accuracy 

grid_search.fit(X_train, y_train) #runs whole seach, training all 27 combos (each 5 times so 135 total runs)

print("Best parameters:", grid_search.best_params_) #.best_params_ tells you which specific combination of n_estimators, max_depth, min_samples_split performs best across all cross-validation tests
print("Best cross-validation accuracy:", grid_search.best_score_) #.best_score_ shows average accuracy that winning combination achieved across cros-validation

best_model = grid_search.best_estimator_ #.best_estimator_ gives the actual trained model object that used the winning setting

turned_predictions = best_model.predict(X_test) #uses best-turned model to make predictions on X_test

turned_accuracy = accuracy_score(y_test, turned_predictions)
print("Turned Random Forest accuracy on validation set:", turned_accuracy) #compares predictions against real answers y_test, gets final accuracy score

Best parameters: {'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 300}
Best cross-validation accuracy: 0.8328277356446371
Turned Random Forest accuracy on validation set: 0.8156424581005587


In [13]:
print(pred_columns)

['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'FamilySize', 'Sex_male', 'Embarked_Q', 'Embarked_S', 'Title_Miss', 'Title_Mr', 'Title_Mrs', 'Title_Rare']


In [14]:
final_model = RandomForestClassifier(random_state=42) #creates new random forest model, untrained with same settings as original

final_model.fit(X, y) #trains new model, learns from all the data  

test_predictions = final_model.predict(test[pred_columns]) #test[pred_columns] selects same 13 columns from real kaggle test set used to train the model, .predict() runs trained model on real test data, producing 0 or 1 for every passenger stored in test_prediction

submission = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Survived': test_predictions
}) #builds new small table with 2 columns

submission.to_csv('submission.csv', index=False) #saves table, index=False stops pands from adding its own automatic row-number columns
submission.head()

,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,1
4,896,0
